In [23]:
pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [24]:
from dotenv import load_dotenv
import os
from pathlib import Path
import os, json
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm

from sklearn.metrics import accuracy_score
load_dotenv()

print("Key geladen?", "GEMINI_API_KEY" in os.environ)

Key geladen? True


In [32]:
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-2.0-flash"

In [33]:
PROJECT_ROOT = Path.cwd().parents[1]
DATA_PATH = PROJECT_ROOT / "files"

PROJECT_ROOT = Path.cwd().parents[1]
DATA_PATH = PROJECT_ROOT / "files"

with open(DATA_PATH / "linkedin-cvs-annotated.json", "r", encoding="utf-8") as f:
    cvs = json.load(f)

jobs_active = []
for person_id, cv in enumerate(cvs):
    for job in cv:
        if job["status"] == "ACTIVE":
            jobs_active.append({**job, "person_id": person_id})

df_active = pd.DataFrame(jobs_active)

df_active[["position", "organization", "seniority"]].head(10)

,position,organization,seniority
0,Prokurist,Depot4Design GmbH,Management
1,CFO,Depot4Design GmbH,Management
2,Betriebswirtin,Depot4Design GmbH,Professional
3,Prokuristin,Depot4Design GmbH,Management
4,CFO,Depot4Design GmbH,Management
5,Solutions Architect,Computer Solutions,Professional
6,Medizintechnik Beratung,Udo Weber,Professional
7,Director expansión de negocio.,Grupo Viajes Kontiki.,Director
8,Gerente comercial,Air & Ground Operations Consultancy,Lead
9,Administrador Unico,Viajes Oceano S.L.,Professional


In [34]:
SENIORITY_LABELS = [
    "Junior",
    "Professional",
    "Senior",
    "Lead",
    "Management",
    "Director"
]


In [35]:
SYSTEM = (
    "You are a classifier. "
    f"Return exactly ONE label from: {', '.join(SENIORITY_LABELS)}. "
    "Output ONLY the label."
)

row = df_active.iloc[8]

prompt = f"""Job title: {row["position"]}
Company: {row["organization"]}
Which seniority label applies?
"""

resp = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={"system_instruction": SYSTEM}
)

print(resp.text)

Management



In [ ]:
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score

preds = []

for _, r in tqdm(df_active.iterrows(), total=len(df_active)):
    prompt = f"""Job title: {r["position"]}
Company: {r["organization"]}
Which seniority label applies best? Choose from: {", ".join(SENIORITY_LABELS)}
"""
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"system_instruction": SYSTEM}
    )
    preds.append(resp.text.strip())

df_active["seniority_pred"] = preds

acc = accuracy_score(df_active["seniority"], df_active["seniority_pred"])
print("Accuracy:", round(acc, 4))

/Users/lennartredlich/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 623/623 [04:41<00:00,  2.21it/s]

Accuracy: 0.549


In [37]:
cm = pd.crosstab(
    df_active["seniority"],
    df_active["seniority_pred"],
    normalize="index"
).round(3)

cm

seniority_pred,Director,Junior,Lead,Management,Professional,Senior
seniority,,,,,,
Director,1.000,0.000,0.000,0.000,0.000,0.000
Junior,0.000,0.583,0.000,0.000,0.333,0.083
Lead,0.248,0.008,0.216,0.312,0.208,0.008
Management,0.557,0.000,0.000,0.422,0.010,0.010
Professional,0.014,0.056,0.000,0.157,0.741,0.032
Senior,0.068,0.000,0.000,0.136,0.045,0.750


In [39]:
SENIORITY_LABELS = ["Junior", "Professional", "Senior", "Lead", "Management", "Director"]
rank = {lab: i for i, lab in enumerate(SENIORITY_LABELS)}

y_true = df_active["seniority"]
y_pred = df_active["seniority_pred"]

true_rank = y_true.map(rank)
pred_rank = y_pred.map(rank)

dist = (true_rank - pred_rank).abs()

acc_exact = (dist == 0).mean()
acc_plusminus1 = (dist <= 1).mean()

print("Exact accuracy:", round(acc_exact, 4))
print("Ordinal accuracy (+-1):", round(acc_plusminus1, 4))

Exact accuracy: 0.549
Ordinal accuracy (+-1): 0.825
